In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import wandb
wandb.login(key="25a61b596081a070a75d3cfc16bfad96632650fc")

In [ ]:
config={
"base_learning_rate": 0.0001,
"label_smoothing": 0.1,
"epochs": 10,
"layers": 12,
"frames": 8,
"val_epochs": 2,
"batch_size": 16,
"load_run": False
}

In [ ]:
wandb.init(
    project="msc",
    config=config
)

In [ ]:
wandb.define_metric("epoch_nr")
wandb.define_metric("train_accuracy_step")
wandb.define_metric("train_accuracy_epoch", step_metric="epoch_nr")
wandb.define_metric("valid_accuracy_epoch", step_metric="epoch_nr")
wandb.define_metric("valid_loss_epoch", step_metric="epoch_nr")
wandb.define_metric("train_loss_step")
wandb.define_metric("train_loss_epoch", step_metric="epoch_nr")

In [ ]:
from torchvision.datasets import HMDB51
import torch
import matplotlib.pyplot as plt
import tqdm
import torch
from utils import save_run, load_run

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
def show_img(image):
    plt.imshow(image.permute(1,2,0).detach().cpu().numpy()*0.2+0.42)
    plt.show()

In [ ]:
from transforms import Div255
from torchvision.transforms.v2 import ScaleJitter, RandAugment, RandomCrop, Compose, ColorJitter, RandomHorizontalFlip, Normalize, RandomApply, UniformTemporalSubsample, CenterCrop, Lambda

ds_transform_train = Compose(transforms=[
    RandomHorizontalFlip(p=0.5),
    ScaleJitter(target_size=(240, 320), scale_range=(0.9, 1.33)),
    RandomCrop(size=(224,224), pad_if_needed=True, fill=128),
    RandomApply(p=0.8, transforms=[ColorJitter(brightness=0.2, contrast=0.1, saturation=0.1, hue=0.1)]),
    RandAugment(num_ops=2, magnitude=10, fill=128),
    Div255(),
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ]
)

ds_transform_valid = Compose(transforms=[
    UniformTemporalSubsample(num_samples=config["frames"]),
    Div255(),
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ]
)

In [ ]:
dataset_train = HMDB51(root="datasets/hmdb/data",
                 annotation_path="datasets/hmdb/test_train_splits/testTrainMulti_7030_splits",
                 frames_per_clip=60,
                 step_between_clips=30,
                 num_workers=4,
                 transform=ds_transform_train,
                 train=True,
                 output_format="TCHW")


dataset_valid = HMDB51(root="datasets/hmdb/data",
                 annotation_path="datasets/hmdb/test_train_splits/testTrainMulti_7030_splits",
                 frames_per_clip=60,
                 step_between_clips=30,
                 num_workers=4,
                 transform=ds_transform_train,
                 train=False,
                 output_format="TCHW")

In [92]:
from pytorchvideo.data.hmdb51 import Hmdb51
from pytorchvideo.data.clip_sampling import UniformClipSampler
from pytorchvideo.transforms import UniformTemporalSubsample, ApplyTransformToKey
from constants import HMDB_CLASSES
from transforms import SplitEqually


tf = Compose(
            [
            ApplyTransformToKey(
              key="video",
              transform=Compose(
                  [
                    Lambda(lambda x: x.type(torch.uint8)),
                    Lambda(lambda x: x.permute(1, 0, 2, 3)),
                    SplitEqually(num_frames=64),
                    UniformTemporalSubsample(num_samples=8, temporal_dim=-4),
                    CenterCrop(size=(224,224)),
                    Div255(),
                    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                  ]
                ),
              ),
              ApplyTransformToKey(
                    key="label",
                    transform=Compose(
                        [
                            lambda x: HMDB_CLASSES.index(x)
                        ]
                    )
              )
            ]
        )

dataset_test = Hmdb51(data_path=r"datasets\hmdb\test_train_splits\testTrainMulti_7030_splits",
                      clip_sampler=UniformClipSampler(1000),  # sample whole clips
                      decode_audio=False,
                      video_path_prefix="datasets/hmdb/data",
                      split_type="test",
                      transform=tf
                      )



In [ ]:
from transformers import ViTForImageClassification
vit = ViTForImageClassification.from_pretrained("WinKawaks/vit-small-patch16-224")

In [ ]:
from transformers import TimesformerConfig, TimesformerForVideoClassification
timesformer_config = TimesformerConfig(num_frames=config["frames"], hidden_size=384, num_attention_heads=6, intermediate_size=384*4, num_labels=51, num_hidden_layers=config["layers"])
timesformer = TimesformerForVideoClassification(timesformer_config)
timesformer.to(device)

In [ ]:
with torch.no_grad():
    timesformer.timesformer.embeddings.patch_embeddings.projection.load_state_dict(vit.vit.embeddings.patch_embeddings.projection.state_dict())
    timesformer.timesformer.embeddings.position_embeddings.copy_(vit.vit.embeddings.position_embeddings)
    #timesformer.timesformer.embeddings.time_embeddings
    for i in range(config["layers"]):
        vit_qkv_weight = torch.cat([vit.vit.encoder.layer[i].attention.attention.query.weight, vit.vit.encoder.layer[i].attention.attention.key.weight, vit.vit.encoder.layer[i].attention.attention.value.weight])
        vit_qkv_bias = torch.cat([vit.vit.encoder.layer[i].attention.attention.query.bias, vit.vit.encoder.layer[i].attention.attention.key.bias, vit.vit.encoder.layer[i].attention.attention.value.bias])

        timesformer.timesformer.encoder.layer[i].attention.attention.qkv.weight.copy_(vit_qkv_weight)
        timesformer.timesformer.encoder.layer[i].attention.attention.qkv.bias.copy_(vit_qkv_bias)
        timesformer.timesformer.encoder.layer[i].attention.output.dense.load_state_dict(vit.vit.encoder.layer[i].attention.output.dense.state_dict())
        timesformer.timesformer.encoder.layer[i].intermediate.dense.load_state_dict(vit.vit.encoder.layer[i].intermediate.dense.state_dict())
        timesformer.timesformer.encoder.layer[i].output.dense.load_state_dict(vit.vit.encoder.layer[i].output.dense.state_dict())
        timesformer.timesformer.encoder.layer[i].layernorm_before.load_state_dict(vit.vit.encoder.layer[i].layernorm_before.state_dict())
        timesformer.timesformer.encoder.layer[i].layernorm_after.load_state_dict(vit.vit.encoder.layer[i].layernorm_after.state_dict())
        timesformer.timesformer.encoder.layer[i].temporal_layernorm.load_state_dict(vit.vit.encoder.layer[i].layernorm_before.state_dict())
        timesformer.timesformer.encoder.layer[i].temporal_attention.attention.qkv.weight.copy_(vit_qkv_weight)
        timesformer.timesformer.encoder.layer[i].temporal_attention.attention.qkv.bias.copy_(vit_qkv_bias)
        timesformer.timesformer.encoder.layer[i].temporal_attention.output.dense.load_state_dict(vit.vit.encoder.layer[i].attention.output.dense.state_dict())
        #timesformer.timesformer.encoder.layer[i].temporal_dense
    timesformer.timesformer.layernorm.load_state_dict(vit.vit.layernorm.state_dict())

In [35]:
from torch.utils.data import DataLoader

training_loader = DataLoader(dataset=dataset_train, batch_size=16)
validation_loader = DataLoader(dataset=dataset_valid, batch_size=4)

In [ ]:
k["video_name"]

'Updatedvideoofmeshootingteamusaarchery_shoot_bow_u_nm_np1_fr_med_3.avi'

In [37]:
for k in training_loader:
      print(k["video_name"])
      break

c:\Users\wgierulski\AppData\Local\anaconda3\Lib\site-packages\torchvision\transforms\functional.py:1603: UserWarning: The default value of the antialias parameter of all the resizing transforms (Resize(), RandomResizedCrop(), etc.) will change from None to True in v0.17, in order to be consistent across the PIL and Tensor backends. To suppress this warning, directly pass antialias=True (recommended, future default), antialias=None (current default, which means False for Tensors and True for PIL), or antialias=False (only works on Tensors - PIL will still use antialiasing). This also applies if you are using the inference transforms from the models weights: update the call to weights.transforms(antialias=True).
  warnings.warn(


RuntimeError: stack expects each tensor to be equal size, but got [30, 3, 224, 224] at entry 0 and [29, 3, 224, 224] at entry 1

In [ ]:
from models.vivit_custom.utils.scheduler import WarmupCosineSchedule
epochs = config["epochs"]
label_smoothing = config["label_smoothing"]

optimizer = torch.optim.SGD(timesformer.parameters(), lr=config["base_learning_rate"], momentum=0.9, weight_decay=1e-5)
#scheduler = WarmupCosineSchedule(optimizer, warmup_steps=int(0.06 * len(training_loader)), t_total=len(training_loader) * epochs)
loss = torch.nn.functional.cross_entropy

In [ ]:
from torchvision.transforms.v2 import MixUp
mixup = MixUp(alpha=0.1, num_classes=51).to(device)

In [ ]:
if config["load_run"]:
      timesformer, optimizer, prev_epoch = load_run(timesformer, optimizer, config["load_run"])
else:
      prev_epoch = 0

In [ ]:
for epoch in range(1 + prev_epoch, 1 + prev_epoch + epochs):
    timesformer.train()
    print(f"Current epoch: {epoch}")
    curr_epoch_losses_train = []
    curr_epoch_accuracies_valid = []
    total_epoch_samples = 0
    total_epoch_correct_train = 0
    for i, data in enumerate(tqdm.tqdm(training_loader)):
        images, _, labels = data
        images = images.to(device)
        labels = labels.to(device)

        #images, labels_mixed_up = mixup(images, labels)

        optimizer.zero_grad()

        outputs = timesformer(images).logits

        loss_train = loss(outputs, labels, label_smoothing=label_smoothing)
        loss_train.backward()

        # logging
        _, predicted = torch.max(outputs, dim=1)
        total_epoch_samples += len(images)
        correct = (predicted == labels).sum().item()
        total_epoch_correct_train += correct
        step_accuracy = correct / len(images)
        curr_epoch_losses_train.append(loss_train.item())
        wandb.log({"train_accuracy_step": step_accuracy, "train_loss_step": loss_train.item()})

        #scheduler.step()
        optimizer.step()
    save_run(timesformer, optimizer, wandb.run.name, epoch)
    
    # logging
    wandb.log({
        "train_accuracy_epoch": total_epoch_correct_train / total_epoch_samples,
        "train_loss_epoch": sum(curr_epoch_losses_train) / len(training_loader),
        "epoch_nr": epoch
               })
    
    if epoch % config["val_epochs"] == 0:
        timesformer.eval()
        total_loss = 0
        total_batches = 0
        total_correct = 0
        total_samples = 0
        with torch.no_grad():
            for i, data in enumerate(tqdm.tqdm(validation_loader)):
                images, _, labels = data
                images = images.to(device)
                labels = labels.to(device)

                outputs = timesformer(images).logits

                loss_valid = loss(outputs, labels)

                # logging
                _, predicted = torch.max(outputs, dim=1)
                correct = (predicted == labels).sum().item()
                total_batches += 1
                total_loss += loss_valid.item()
                total_correct += correct
                total_samples += len(images)

        wandb.log({"valid_accuracy_epoch": total_correct / total_samples,
                   "valid_loss_epoch": total_loss / total_batches
                   }
                   )

